# TW Hydrae — ALMA archive query, download and substructure analysis (Julia)

Julia port of `alma_twhya_query_15.py`. Queries the ALMA archive for TW Hya, downloads the Andrews et al. 2016 Band 7 continuum product (project `2015.1.00686.S`), displays the image, builds a deprojected radial profile, and finds rings/gaps by Savitzky–Golay baseline subtraction, with a comparison to Tsukagoshi et al. 2016 / Andrews et al. 2016 gap radii.

**Python → Julia mapping**

| Python | Julia |
|---|---|
| `astroquery.alma` (`query_object`, `get_data_info`) | ALMA TAP (ADQL) + datalink via `HTTP.jl`, parsed with `CSV.jl` / `EzXML.jl` |
| `astropy.io.fits`, `WCS` | `FITSIO.jl` (+ pixel scales from the header) |
| `scipy.signal.savgol_filter`, `find_peaks` | small native ports in this notebook (verified against SciPy) |
| `matplotlib` | `CairoMakie.jl` (dark theme) |
| `urllib`, `tarfile` | `Downloads`, `Tar` (stdlib) |

Run the cells top to bottom. The first run installs packages into `./alma_twhya_env` and precompiles (a few minutes). Kernel: Julia ≥ 1.10 (written for 1.12).

In [ ]:
println("TW Hydrae - Closest known protoplanetary disk to Earth.")
println("https://almascience.nrao.edu/aq/?sourceNameResolver=TW%20Hydrae&skyViewHeight=48.99&skyViewWidth=50.95")

## Setup

In [ ]:
# alma_twhya_query_julia_1.ipynb
#
# Julia port of alma_twhya_query_15.py (Query, download, and display ALMA
# archive data for TW Hydrae).
#
# Changelog
#   v1  Initial Julia port of the Python v15 pipeline: ALMA TAP + datalink
#       query with mirror retry, tarball download/FITS extraction, 2D image
#       display, deprojected radial profile, Savitzky-Golay baseline
#       subtraction, ring/gap detection, literature comparison.
#
# Packages are installed into a local project environment (./alma_twhya_env)
# so the global Julia environment is left untouched. The first run
# downloads and precompiles everything (CairoMakie takes a few minutes).

import Pkg
Pkg.activate(joinpath(pwd(), "alma_twhya_env"))

let needed = ["HTTP", "EzXML", "CSV", "DataFrames", "FITSIO", "CairoMakie"]
    installed = keys(Pkg.project().dependencies)
    to_add = filter(p -> !(p in installed), needed)
    if !isempty(to_add)
        println("Installing: ", join(to_add, ", "))
        Pkg.add(to_add)
    end
end

using HTTP, EzXML, CSV, DataFrames, FITSIO, CairoMakie
using Downloads, Tar                      # stdlib
using Statistics, LinearAlgebra, Printf   # stdlib

println("Julia ", VERSION, " | environment: ", dirname(Base.active_project()))

## Configuration, literature values and plot theme

In [ ]:
const CACHE_DIR  = "./alma_cache"
const OUTPUT_DIR = "./alma_twhya_plots"
mkpath(CACHE_DIR)
mkpath(OUTPUT_DIR)

# Set true to print step-by-step checkpoints inside the radial-profile
# functions -- useful for tracing exactly where execution stops if a cell
# silently produces no output/plot.
const DEBUG = Ref(true)

# Set true to also write every figure to OUTPUT_DIR as a PNG.
const SAVE_PLOTS = Ref(true)

_dbg(args...) = DEBUG[] && println("[DEBUG] ", join(string.(args), " "))

# ALMA's TAP/datalink services intermittently return transient 502/503
# "Proxy Error" responses under load, even for valid queries -- this is
# server-side flakiness, not a query bug. The regional mirrors all serve the
# same archive, so we retry with backoff and rotate mirrors if one is down.
const ALMA_MIRRORS = [
    "https://almascience.nrao.edu",
    "https://almascience.eso.org",
    "https://almascience.nao.ac.jp",
]

# TW Hya ICRS position (SIMBAD). astroquery resolves the name for you;
# there is no equivalent name resolver here, so the position is hardcoded.
const TWHYA_RA_DEG  = 165.466273
const TWHYA_DEC_DEG = -34.704731

# Literature substructure radii for TW Hya, for visual/numeric sanity-checking
# our own detections. Reported by Tsukagoshi et al. 2016 (ApJL 829, L35),
# consistent with Andrews et al. 2016 (ApJL 820, L40). These papers assumed
# a pre-Gaia distance of ~54 pc; scale to whatever distance_pc this pipeline
# used (e.g. the Gaia DR2/DR3 value ~60.1 pc) before comparing, since a
# ~11% distance difference is a ~11% radius difference, not a detection
# discrepancy.
const TSUKAGOSHI2016_TWHYA = (
    distance_pc     = 54.0,
    central_hole_au = 1.0,
    deep_gaps_au    = [22.0, 37.0],
    shallow_gaps_au = [6.0, 28.0, 44.0],
)

"""
    literature_features_au(target_distance_pc) -> Vector{Tuple{Float64,String,Symbol}}

Return TW Hya literature gap radii (Tsukagoshi et al. 2016 / Andrews et al.
2016) rescaled from their assumed distance to `target_distance_pc`, as
`(radius_au, label, kind)` tuples for plotting/comparison. `kind` is one of
`:hole`, `:deep`, `:shallow`.
"""
function literature_features_au(target_distance_pc)
    lit = TSUKAGOSHI2016_TWHYA
    scale = target_distance_pc / lit.distance_pc
    features = [(lit.central_hole_au * scale, "central hole (lit.)", :hole)]
    append!(features, [(r * scale, "deep gap (lit.)", :deep) for r in lit.deep_gaps_au])
    append!(features, [(r * scale, "shallow gap (lit.)", :shallow) for r in lit.shallow_gaps_au])
    return features
end

# Dark, publication-style theme (#111111 background, #EEEEEE text).
const BG_COLOR = "#111111"
const FG_COLOR = "#EEEEEE"
const C_FLUX   = "#FF9F43"   # azimuthal mean
const C_RING   = "#FFD43B"   # rings
const C_GAP    = "#5DADE2"   # gaps
const C_RESID  = "#4EA1FF"   # residual curve
const C_LIT    = "#FF4D6D"   # literature gap markers

function alma_dark_theme!()
    set_theme!(theme_dark())
    update_theme!(
        backgroundcolor = BG_COLOR,
        Axis = (
            backgroundcolor = BG_COLOR,
            xgridcolor = (FG_COLOR, 0.10), ygridcolor = (FG_COLOR, 0.10),
            xlabelcolor = FG_COLOR, ylabelcolor = FG_COLOR, titlecolor = FG_COLOR,
            xticklabelcolor = FG_COLOR, yticklabelcolor = FG_COLOR,
            xtickcolor = FG_COLOR, ytickcolor = FG_COLOR,
            leftspinecolor = FG_COLOR, rightspinecolor = FG_COLOR,
            topspinecolor = FG_COLOR, bottomspinecolor = FG_COLOR,
        ),
        Legend = (bgcolor = BG_COLOR, framecolor = (FG_COLOR, 0.4), labelcolor = FG_COLOR),
        Colorbar = (labelcolor = FG_COLOR, ticklabelcolor = FG_COLOR, tickcolor = FG_COLOR),
    )
end
alma_dark_theme!()

function _save_fig(fig, name)
    SAVE_PLOTS[] || return nothing
    path = joinpath(OUTPUT_DIR, name)
    save(path, fig; px_per_unit = 2)
    println("Saved figure: ", path)
    return path
end

## ALMA archive access (TAP query, datalink, mirror retry)

In [ ]:
# ---------------------------------------------------------------------------
# HTTP helpers: mirror rotation + retry/backoff, TAP queries, datalink
# ---------------------------------------------------------------------------

const TRANSIENT_STATUS = (502, 503, 504)

"""
    robust_get(path; query, max_attempts=3, base_delay=5) -> (response, mirror)

GET `<mirror><path>` with retry/backoff on transient errors (502/503/504,
network exceptions), rotating through `ALMA_MIRRORS`. Non-transient
responses (including 4xx) are returned to the caller to interpret.
"""
function robust_get(path::AbstractString; query = Dict{String,String}(),
                    max_attempts::Int = 3, base_delay::Real = 5)
    last_err = nothing
    for mirror in ALMA_MIRRORS
        for attempt in 1:max_attempts
            println("  [mirror=$mirror, attempt $attempt/$max_attempts] GET $path ...")
            try
                resp = HTTP.get(mirror * path; query = query, status_exception = false,
                                retry = false, readtimeout = 180, connect_timeout = 30)
                if resp.status in TRANSIENT_STATUS
                    last_err = "HTTP $(resp.status)"
                    if attempt < max_attempts
                        delay = base_delay * attempt
                        println("  Transient error ($(last_err)); retrying in $(delay)s ...")
                        sleep(delay)
                        continue
                    end
                    break
                end
                return resp, mirror
            catch err
                last_err = sprint(showerror, err)
                if attempt < max_attempts
                    delay = base_delay * attempt
                    println("  Transient error ($(typeof(err))); retrying in $(delay)s ...")
                    sleep(delay)
                end
            end
        end
        println("  Mirror $mirror did not succeed, trying next mirror ...")
    end
    error("GET $path failed across all mirrors $(ALMA_MIRRORS). Last error: $last_err")
end

# ---- minimal VOTable (TABLEDATA) reader -----------------------------------

"""
    parse_votable(text) -> DataFrame

Parse the first TABLE of a VOTable using the TABLEDATA serialization into an
all-`String` DataFrame. Raises on a server-reported error or on BINARY/BINARY2
serializations (request CSV or a TABLEDATA VOTable instead).
"""
function parse_votable(text::AbstractString)
    # Strip XML namespace declarations so plain XPath works.
    clean = replace(text, r"\s(xmlns(:\w+)?|xsi:\w+)=\"[^\"]*\"" => "")
    doc = EzXML.parsexml(clean)

    err_node = findfirst("//INFO[@name='QUERY_STATUS'][@value='ERROR']", doc)
    if err_node !== nothing
        error("Server reported a query error: ", strip(nodecontent(err_node)))
    end

    table = findfirst("//RESOURCE[@type='results']/TABLE", doc)
    table === nothing && (table = findfirst("//TABLE", doc))
    table === nothing && error("No <TABLE> found in VOTable response.")

    if findfirst("./DATA/BINARY", table) !== nothing || findfirst("./DATA/BINARY2", table) !== nothing
        error("VOTable uses BINARY serialization; only TABLEDATA/CSV are supported here.")
    end

    names = [field["name"] for field in findall("./FIELD", table)]
    rows = findall("./DATA/TABLEDATA/TR", table)
    cols = [String[] for _ in names]
    for tr in rows
        tds = findall("./TD", tr)
        for (k, col) in enumerate(cols)
            push!(col, k <= length(tds) ? String(strip(nodecontent(tds[k]))) : "")
        end
    end
    return DataFrame([Symbol(n) => c for (n, c) in zip(names, cols)])
end

"""
    tap_query(adql) -> DataFrame

Run a synchronous ADQL query against the ALMA TAP service (mirror rotation
and retry included). Tries `FORMAT=csv` first and falls back to a VOTable
if the server does not honour it. All columns are returned as `String`.
"""
function tap_query(adql::AbstractString)
    last_msg = ""
    for fmt in ("csv", "votable")
        query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL",
                     "FORMAT" => fmt, "QUERY" => adql)
        resp, mirror = robust_get("/tap/sync"; query = query)
        text = String(resp.body)
        _dbg("tap_query: format =", fmt, "| HTTP status =", resp.status,
             "| bytes =", length(text), "| mirror =", mirror)
        try
            resp.status == 200 || error("HTTP $(resp.status): ", first(text, 400))
            if startswith(lstrip(text), "<")
                return parse_votable(text)
            else
                df = CSV.read(IOBuffer(text), DataFrame; types = String, silencewarnings = true)
                return mapcols(c -> coalesce.(c, ""), df)
            end
        catch err
            last_msg = sprint(showerror, err)
            println("  FORMAT=$fmt failed: ", first(last_msg, 300))
        end
    end
    error("TAP query failed in both CSV and VOTable formats. Last error: $last_msg")
end

"""
    robust_get_data_info(member_ous_uid) -> DataFrame

Datalink listing of the downloadable products for one member OUS (the
equivalent of astroquery's `Alma.get_data_info`). Returns the datalink
table with empty-URL rows removed and URLs percent-decoded.

The ALMA datalink service only exposes whole `.tar` packages for this kind
of delivery, so astroquery's `expand_tarfiles=True` returns the same
listing and no separate option is needed here.
"""
function robust_get_data_info(member_ous_uid::AbstractString)
    println("  fetching data info for $member_ous_uid ...")
    resp, mirror = robust_get("/datalink/sync"; query = Dict("ID" => String(member_ous_uid)))
    resp.status == 200 || error("datalink returned HTTP $(resp.status): ", first(String(resp.body), 400))
    df = parse_votable(String(resp.body))
    _dbg("robust_get_data_info: columns =", names(df), "| rows =", nrow(df), "| mirror =", mirror)
    "access_url" in names(df) || error("datalink response has no access_url column; columns = $(names(df))")
    df = df[.!isempty.(df.access_url), :]
    df.access_url = [HTTP.URIs.unescapeuri(u) for u in df.access_url]
    return df
end

## Query → inspect → pick product tarball

In [ ]:
# ---------------------------------------------------------------------------
# Query -> inspect -> pick product
# ---------------------------------------------------------------------------

"""
    query_tw_hya(; public=true, science=true, proposal_id=nothing, radius_deg=0.05) -> DataFrame

Query the ALMA archive (ivoa.obscore via TAP/ADQL) for TW Hydrae
observations. Like astroquery's `query_object`, the server-side selection is
a loose sky-position match (any footprint intersecting a cone around TW Hya)
which can pull in nearby/related fields, so the rows are then masked down to
those that actually target TW Hya by name.

proposal_id: if given (e.g. "2015.1.00686.S"), further restrict results to
    that specific proposal -- useful for grabbing the well-known Andrews et
    al. 2016 high-resolution Band 7 ring image rather than whatever UID
    happens to come up first among TW Hya's ~80 observations.
"""
function query_tw_hya(; public::Bool = true, science::Bool = true,
                      proposal_id::Union{Nothing,AbstractString} = nothing,
                      radius_deg::Real = 0.05)
    conds = ["INTERSECTS(CIRCLE('ICRS', $TWHYA_RA_DEG, $TWHYA_DEC_DEG, $radius_deg), s_region) = 1"]
    public  && push!(conds, "data_rights = 'Public'")
    science && push!(conds, "science_observation = 'T'")
    proposal_id === nothing || push!(conds, "proposal_id = '$(strip(proposal_id))'")

    select = "SELECT obs_id, target_name, member_ous_uid, band_list, frequency, " *
             "spatial_resolution, velocity_resolution, t_exptime, proposal_id, " *
             "obs_release_date, science_keyword FROM ivoa.obscore WHERE "
    println("Querying ALMA archive for source: TW Hydrae ...")
    _dbg("query_tw_hya: ADQL =", select * join(conds, " AND "))

    results = try
        tap_query(select * join(conds, " AND "))
    catch err
        # If the data_rights / science_observation columns are rejected, retry
        # with the position (and proposal) constraints only.
        println("Query with public/science filters failed (", first(sprint(showerror, err), 200),
                "); retrying with position", proposal_id === nothing ? "" : " + proposal", " only ...")
        base = [conds[1]]
        proposal_id === nothing || push!(base, last(conds))
        tap_query(select * join(base, " AND "))
    end
    println("Found $(nrow(results)) raw rows (a position query can return nearby/related fields too).")

    name_col = "target_name" in names(results) ? :target_name : :obs_id
    keep = map(results[!, name_col]) do val
        n = lowercase(replace(string(val), " " => "_"))
        occursin("tw_hya", n) || occursin("twhya", n)
    end
    results = results[keep, :]
    println("Filtered to $(nrow(results)) rows matching TW Hydrae.")

    if proposal_id !== nothing && "proposal_id" in names(results)
        pkeep = [uppercase(strip(string(v))) == uppercase(strip(proposal_id)) for v in results.proposal_id]
        results = results[pkeep, :]
        println("Further filtered to $(nrow(results)) rows for proposal $proposal_id.")
    end

    println()
    cols = filter(c -> c in names(results),
                  ["obs_id", "target_name", "member_ous_uid", "band_list", "frequency",
                   "spatial_resolution", "velocity_resolution", "t_exptime",
                   "proposal_id", "obs_release_date", "science_keyword"])
    show(stdout, MIME("text/plain"), results[:, cols]; allcols = true, allrows = true, truncate = 40)
    println()
    return results
end

function _to_gb(s)
    x = tryparse(Float64, s)
    return x === nothing ? missing : round(x / 1e9; digits = 3)
end

"""
    inspect_products(member_ous_uid) -> DataFrame

List the individual data files (here: tar packages) available for one ALMA
project (member OUS UID), with sizes in GB.
"""
function inspect_products(member_ous_uid::AbstractString)
    println("\nFetching file listing for $member_ous_uid ...")
    data_info = robust_get_data_info(member_ous_uid)
    if "content_length" in names(data_info)
        data_info.size_GB = [_to_gb(s) for s in data_info.content_length]
    end
    cols = filter(c -> c in names(data_info), ["access_url", "content_type", "size_GB"])
    show(stdout, MIME("text/plain"), data_info[:, cols]; allcols = true, allrows = true, truncate = 110)
    println()
    return data_info
end

"""
    pick_product_tarball(data_info) -> Union{String,Nothing}

Some ALMA deliveries (like this TW Hya Band 7 project) don't expose
individual FITS files through the datalink listing at all -- only whole
`.tar` packages: the main pipeline-products tar, an "auxiliary" tar, raw ASDM
tars, and an "external_ari_l" tar (reprocessed ARI-L products). In that case
we can't filter by ".fits" URL suffix; we have to pick the right tarball and
extract FITS members from it after downloading.

Preference order:
  1. The plain main product tar (calibrated pipeline images):
     "<proposal>_uid___...__NNN_of_NNN.tar", excluding auxiliary/asdm.
  2. The external_ari_l tar (ARI-L reprocessed images), as a fallback.
Raw ASDM tars and the auxiliary tar are skipped -- they hold uncalibrated or
ancillary data, not science-ready images, and are much larger than needed
just to grab a continuum image.
"""
function pick_product_tarball(data_info::DataFrame)
    urls = String.(data_info.access_url)

    is_main_product_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && !occursin("auxiliary", lu) && !occursin("asdm", lu) &&
            !occursin("external_ari_l", lu) && !occursin("readme", lu)
    end
    is_ari_l_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && occursin("external_ari_l", lu)
    end

    main_candidates = filter(is_main_product_tar, urls)
    isempty(main_candidates) || return first(main_candidates)

    ari_l_candidates = filter(is_ari_l_tar, urls)
    if !isempty(ari_l_candidates)
        println("No plain main-product tar found; falling back to the " *
                "external_ari_l (reprocessed) tarball.")
        return first(ari_l_candidates)
    end

    println("No suitable product tarball found -- only auxiliary/raw ASDM " *
            "tars are available for this UID. Inspect data_info manually.")
    return nothing
end

## Download tarball and extract FITS members

In [ ]:
# ---------------------------------------------------------------------------
# Download a product tarball and extract only the FITS members
# ---------------------------------------------------------------------------

_find_fits(dir; pattern = r"\.fits$"i) =
    isdir(dir) ? [joinpath(root, f) for (root, _, files) in walkdir(dir)
                  for f in files if occursin(pattern, f)] : String[]

"""
    download_and_extract_fits(tarball_url; dest_dir=CACHE_DIR, fits_regex, delete_tar=true) -> Vector{String}

Download a single ALMA product tarball and extract only the FITS members
from it, then delete the tarball (ALMA only exposes whole `.tar` packages,
which can be multiple GB, so we don't keep them). Skips everything if FITS
files were already extracted on a previous run.
"""
function download_and_extract_fits(tarball_url::AbstractString; dest_dir::AbstractString = CACHE_DIR,
                                   fits_regex::Regex = r"\.fits$"i, delete_tar::Bool = true)
    tarball_name = basename(tarball_url)
    tar_local = joinpath(dest_dir, tarball_name)
    extract_dir = joinpath(dest_dir, splitext(tarball_name)[1] * "_extracted")

    existing = _find_fits(extract_dir; pattern = fits_regex)
    if !isempty(existing)
        println("Already have $(length(existing)) extracted FITS file(s), skipping download.")
        return existing
    end

    if !isfile(tar_local)
        println("\nDownloading $tarball_url ...")
        part = tar_local * ".part"          # never leave a truncated .tar that looks complete
        last_pct = Ref(-10)
        Downloads.download(tarball_url, part; progress = (total, now) -> begin
            total > 0 || return
            pct = floor(Int, 100 * now / total)
            if pct >= last_pct[] + 10
                last_pct[] = pct
                @printf("  %3d%%  (%.2f / %.2f GB)\n", pct, now / 1e9, total / 1e9)
            end
        end)
        mv(part, tar_local; force = true)
        println("Download complete.")
    else
        println("Found cached tarball at $tar_local.")
    end

    println("Extracting FITS members ...")
    headers = Tar.list(tar_local; strict = false)
    members = String[h.path for h in headers if h.type == :file && occursin(fits_regex, h.path)]
    if isempty(members)
        println("No FITS members found inside this tarball.")
    else
        wanted = Set(members)
        isdir(extract_dir) && rm(extract_dir; recursive = true, force = true)   # Tar.extract needs an empty target
        try
            Tar.extract(h -> h.path in wanted, tar_local, extract_dir)
        catch err
            println("Tar.extract failed: ", sprint(showerror, err))
        end
        if !all(m -> isfile(joinpath(extract_dir, m)), members)
            println("Pure-Julia extraction incomplete; falling back to system `tar` ...")
            mkpath(extract_dir)
            run(`tar -xf $tar_local -C $extract_dir $members`)
        end
    end
    extracted_paths = [joinpath(extract_dir, m) for m in members if isfile(joinpath(extract_dir, m))]

    # Only delete the (large) tarball once we actually have the FITS files.
    if delete_tar && !isempty(extracted_paths) && isfile(tar_local)
        rm(tar_local)
        println("Deleted $tarball_name after extraction to save space.")
    end
    return extracted_paths
end

## FITS loading and 2D image display

In [ ]:
# ---------------------------------------------------------------------------
# FITS loading + 2D image display
# ---------------------------------------------------------------------------

_hget(h, key, default = nothing) = haskey(h, key) ? h[key] : default

"""
    pixel_scales_arcsec(header) -> (cdelt1_arcsec, cdelt2_arcsec)

Signed pixel scales along the two sky axes, from CDELT1/2 or (if absent) the
CD matrix. RA normally increases to the left, so `cdelt1` is negative.
"""
function pixel_scales_arcsec(h)
    if haskey(h, "CDELT1") && haskey(h, "CDELT2")
        return 3600.0 * h["CDELT1"], 3600.0 * h["CDELT2"]
    elseif haskey(h, "CD1_1")
        cd11, cd12 = h["CD1_1"], _hget(h, "CD1_2", 0.0)
        cd21, cd22 = _hget(h, "CD2_1", 0.0), h["CD2_2"]
        s1, s2 = hypot(cd11, cd21), hypot(cd12, cd22)
        flip = (cd11 * cd22 - cd12 * cd21) < 0      # standard sky orientation
        return 3600.0 * (flip ? -s1 : s1), 3600.0 * s2
    else
        error("FITS header has neither CDELT1/2 nor a CD matrix; can't determine pixel scale.")
    end
end

"""
    load_fits_image(filepath) -> NamedTuple

Read the first image HDU with >= 2 axes, collapse any trailing axes
(frequency/Stokes) to their first plane, and return the 2D array indexed
`[x, y]` (FITS/Julia order) together with the header values this notebook
needs.
"""
function load_fits_image(filepath::AbstractString)
    f = FITS(filepath)
    try
        idx = findfirst(i -> f[i] isa ImageHDU && ndims(f[i]) >= 2, 1:length(f))
        idx === nothing && error("No image HDU with data found in $filepath")
        hdu = f[idx]
        raw = read(hdu)
        h = read_header(hdu)

        arr = raw
        while ndims(arr) > 2
            arr = selectdim(arr, ndims(arr), 1)
        end
        data = Array(arr)

        cdelt1, cdelt2 = pixel_scales_arcsec(h)
        return (
            data = data,
            cdelt1_arcsec = cdelt1,
            cdelt2_arcsec = cdelt2,
            pix_scale_arcsec = abs(cdelt1),
            bmaj_deg = _hget(h, "BMAJ"),
            bmin_deg = _hget(h, "BMIN"),
            bpa_deg  = _hget(h, "BPA"),
        )
    finally
        close(f)
    end
end

"""
    display_fits_image(filepath; cutout_arcsec=6.0, vmax_percentile=99.5, asinh_a=0.01) -> Figure

Display a 2D (or 2D-slice of a) FITS image with sky-offset axes.

cutout_arcsec: size (in arcsec) of a square region to crop around the image
    center before display.
vmax_percentile: central percentile interval used for the color limits
    (astropy's PercentileInterval semantics: 99.5 clips 0.25% at each end).
    TW Hya's compact central source is much brighter than the surrounding
    disk rings, so a percentile interval keeps vmax below the true peak so
    ring structure doesn't wash out.
asinh_a: softening parameter for the asinh stretch,
    y = asinh(x/a) / asinh(1/a) on the normalized image. Smaller values
    compress the bright core harder, pulling up faint disk/ring structure
    relative to it. Try 0.005-0.05.

TW Hya's disk is only ~1-2" in radius at ~60 pc, and its rings (as first
resolved at high resolution by Andrews et al. 2016) are on sub-arcsec
scales, so a tight cutout (a few arcsec) is needed to actually resolve the
ring structure rather than showing empty sky.

Axes are offsets from the cutout center in arcsec (east to the left) rather
than full RA/Dec WCS ticks.
"""
function display_fits_image(filepath::AbstractString; cutout_arcsec = 6.0,
                            vmax_percentile = 99.5, asinh_a = 0.01)
    img = load_fits_image(filepath)
    data = img.data
    nx, ny = size(data)
    cx, cy = nx ÷ 2 + 1, ny ÷ 2 + 1        # 1-based equivalent of numpy's (nx//2, ny//2)

    ilo, ihi, jlo, jhi = 1, nx, 1, ny
    if cutout_arcsec !== nothing
        size_pix = max(round(Int, cutout_arcsec / img.pix_scale_arcsec), 10)
        half = size_pix ÷ 2
        ilo = max(cx - half, 1);  ihi = min(ilo + size_pix - 1, nx)
        jlo = max(cy - half, 1);  jhi = min(jlo + size_pix - 1, ny)
    end
    cut = Float64.(data[ilo:ihi, jlo:jhi])

    vals = filter(isfinite, vec(cut))
    tail = (100 - vmax_percentile) / 200
    vmin, vmax = quantile(vals, [tail, 1 - tail])
    stretch(x) = asinh(x / asinh_a) / asinh(1 / asinh_a)
    normed = stretch.(clamp.((cut .- vmin) ./ (vmax - vmin), 0.0, 1.0))

    xs = range((ilo - cx) * img.cdelt1_arcsec, step = img.cdelt1_arcsec, length = size(cut, 1))
    ys = range((jlo - cy) * img.cdelt2_arcsec, step = img.cdelt2_arcsec, length = size(cut, 2))

    # heatmap! wants increasing coordinates; flip data if an axis runs backwards
    if img.cdelt1_arcsec < 0
        xs = reverse(xs);  normed = reverse(normed; dims = 1)
    end
    if img.cdelt2_arcsec < 0
        ys = reverse(ys);  normed = reverse(normed; dims = 2)
    end

    fig = Figure(size = (760, 700))
    ax = Axis(fig[1, 1]; aspect = DataAspect(), xreversed = true,
              xlabel = "ΔRA [arcsec]", ylabel = "ΔDec [arcsec]",
              title = "TW Hydrae — $(basename(filepath))", titlesize = 12)
    hm = heatmap!(ax, xs, ys, normed; colormap = :inferno, colorrange = (0, 1))

    tick_vals = collect(range(vmin, vmax; length = 6))
    tick_pos = stretch.((tick_vals .- vmin) ./ (vmax - vmin))
    Colorbar(fig[1, 2], hm; label = "Flux density [Jy/beam]",
             ticks = (tick_pos, [@sprintf("%.1e", v) for v in tick_vals]))

    display(fig)
    _save_fig(fig, "twhya_continuum_image.png")
    return fig
end

## Deprojected radial profile

In [ ]:
# ---------------------------------------------------------------------------
# Deprojected azimuthally averaged radial profile
# ---------------------------------------------------------------------------

"""
    extract_radial_profile(filepath; incl_deg=7.0, pa_deg=155.0, distance_pc=60.1,
                           max_radius_arcsec=3.0, n_bins=60, center=nothing) -> NamedTuple

Azimuthally averaged (deprojected) radial flux profile of a disk image.

TW Hya is viewed nearly face-on, so this is much simpler than the HL Tau
case: inclination is small (i ~ 7 deg, Andrews et al. 2016); PA is poorly
constrained at such low inclination -- ~155 deg is a commonly quoted
literature value, but deprojection barely matters here since cos(7 deg) ~
0.99). Defaults are approximate literature values, not a fit to this
specific image -- override `incl_deg`/`pa_deg` if you have better
constraints (e.g. from a Keplerian rotation fit to the CO cube also
extracted from this tarball).

incl_deg, pa_deg: disk inclination and position angle (deg), used to
    deproject the image into a face-on frame before radial binning.
distance_pc: system distance, used to convert arcsec to au.
max_radius_arcsec: outer radius of the profile.
n_bins: number of radial annuli.
center: `(x, y)` 1-based pixel coordinates of the star. If `nothing`, uses
    the brightest pixel in the image (fine for a centrally-peaked continuum
    image like this one).

The returned NamedTuple carries radius_arcsec, radius_au, mean_flux,
std_flux, the pixel scale, center and beam, so it can be re-plotted or fed
into peak-finding without re-reading the FITS file.
"""
function extract_radial_profile(filepath::AbstractString; incl_deg = 7.0, pa_deg = 155.0,
                                distance_pc = 60.1, max_radius_arcsec = 3.0,
                                n_bins::Int = 60, center = nothing)
    img = load_fits_image(filepath)
    data = img.data
    nx, ny = size(data)
    _dbg("extract_radial_profile: loaded", filepath)
    _dbg("extract_radial_profile: data shape", size(data), "eltype", eltype(data),
         "nan count", count(isnan, data), "finite count", count(isfinite, data))

    # Synthesized beam size (BMAJ/BMIN are in degrees per FITS convention).
    # Any "feature" at a radius comparable to or smaller than the beam FWHM
    # is very likely the PSF of the unresolved central source, not real disk
    # substructure, and should be excluded from peak-finding.
    bmaj_arcsec = bmin_arcsec = bmaj_au = bmin_au = nothing
    if img.bmaj_deg !== nothing && img.bmin_deg !== nothing
        bmaj_arcsec = img.bmaj_deg * 3600.0
        bmin_arcsec = img.bmin_deg * 3600.0
        bmaj_au = bmaj_arcsec * distance_pc
        bmin_au = bmin_arcsec * distance_pc
        @printf("Synthesized beam: %.4f\" x %.4f\" (PA=%s) = %.2f au x %.2f au at %s pc\n",
                bmaj_arcsec, bmin_arcsec, img.bpa_deg, bmaj_au, bmin_au, distance_pc)
        _dbg("extract_radial_profile: BMAJ/BMIN/BPA (deg) =", img.bmaj_deg, img.bmin_deg, img.bpa_deg)
    else
        println("WARNING: no BMAJ/BMIN found in this FITS header -- beam size " *
                "unknown, can't distinguish beam-scale artifacts from real " *
                "substructure near the core by beam size alone.")
    end

    pix = img.pix_scale_arcsec
    _dbg("extract_radial_profile: pix_scale_arcsec =", pix)

    if center === nothing
        # Brightest pixel -- reasonable for a centrally-peaked disk image
        # like this one, where the star dominates the continuum peak.
        _, ci = findmax(v -> isnan(v) ? -Inf : v, data)
        cx, cy = Tuple(ci)
    else
        cx, cy = center
    end
    _dbg("extract_radial_profile: center pixel (cx, cy), 1-based =", (cx, cy),
         "image shape (nx, ny) =", (nx, ny))

    # Deproject: rotate so the disk's major axis aligns with x, then stretch
    # the (foreshortened) minor axis back out by 1/cos(incl).
    sin_pa, cos_pa = sincos(deg2rad(pa_deg))
    cos_i = cos(deg2rad(incl_deg))
    binw = max_radius_arcsec / n_bins
    n_bins < typemax(Int16) || error("n_bins too large")

    binidx = zeros(Int16, nx, ny)          # 0 = outside the profile / NaN pixel
    sums   = zeros(Float64, n_bins)
    counts = zeros(Int, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        v = data[i, j]
        isnan(v) && continue
        dx = (i - cx) * pix
        dy = (j - cy) * pix
        xr = dx * cos_pa + dy * sin_pa
        yr = (-dx * sin_pa + dy * cos_pa) / cos_i
        r = sqrt(xr * xr + yr * yr)
        r < max_radius_arcsec || continue
        b = floor(Int, r / binw) + 1
        b > n_bins && continue
        binidx[i, j] = b
        sums[b] += v
        counts[b] += 1
    end

    mean_flux = [counts[b] > 0 ? sums[b] / counts[b] : NaN for b in 1:n_bins]
    ssq = zeros(Float64, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        b = binidx[i, j]
        b == 0 && continue
        d = data[i, j] - mean_flux[b]
        ssq[b] += d * d
    end
    std_flux = [counts[b] > 0 ? sqrt(ssq[b] / counts[b]) : NaN for b in 1:n_bins]   # population std, like np.nanstd

    bin_edges = range(0, max_radius_arcsec; length = n_bins + 1)
    bin_centers = [0.5 * (bin_edges[k] + bin_edges[k + 1]) for k in 1:n_bins]

    _dbg("extract_radial_profile: bins =", n_bins, "empty bins =", count(==(0), counts),
         "valid mean_flux entries =", count(!isnan, mean_flux))
    if all(isnan, mean_flux)
        println("WARNING: every radial bin is NaN -- check that max_radius_arcsec " *
                "and the pixel scale are compatible (radius map may not overlap " *
                "the requested bin range at all).")
    end

    result = (
        radius_arcsec = bin_centers,
        radius_au = bin_centers .* distance_pc,
        mean_flux = mean_flux,
        std_flux = std_flux,
        pix_scale_arcsec = pix,
        center_pix = (cx, cy),
        incl_deg = incl_deg,
        pa_deg = pa_deg,
        bmaj_arcsec = bmaj_arcsec,
        bmin_arcsec = bmin_arcsec,
        bmaj_au = bmaj_au,
        bmin_au = bmin_au,
        distance_pc = distance_pc,
    )
    _dbg("extract_radial_profile: returning profile with fields", keys(result))
    return result
end

## Peak finding and ring/gap detection

In [ ]:
# ---------------------------------------------------------------------------
# Signal-processing helpers (ports of scipy.signal.savgol_filter / find_peaks)
# ---------------------------------------------------------------------------

"""
    savgol_filter(y, window, polyorder) -> Vector{Float64}

Savitzky-Golay smoothing equivalent to `scipy.signal.savgol_filter(y,
window, polyorder)` with its default `mode="interp"`: interior points use
the least-squares polynomial fit over the centred window; the first and last
`window÷2` points evaluate the polynomial fitted to the first/last `window`
samples.
"""
function savgol_filter(y::AbstractVector{<:Real}, window::Integer, polyorder::Integer)
    isodd(window) || throw(ArgumentError("window must be odd, got $window"))
    window > polyorder || throw(ArgumentError("window must exceed polyorder"))
    n = length(y)
    window <= n || throw(ArgumentError("window ($window) larger than data length ($n)"))
    h = window ÷ 2
    xs = collect(-h:h)
    V = [Float64(x)^k for x in xs, k in 0:polyorder]    # window × (polyorder+1)
    P = pinv(V)                                         # least-squares solver
    out = zeros(Float64, n)
    for i in 1:n
        s = clamp(i - h, 1, n - window + 1)            # window start (shifted at the edges)
        c = P * Float64.(y[s:(s + window - 1)])
        xi = Float64(i - (s + h))                       # position of i relative to the window centre
        out[i] = sum(c[k + 1] * xi^k for k in 0:polyorder)
    end
    return out
end

# Local maxima, handling flat plateaus by taking the middle sample
# (scipy.signal._peak_finding_utils._local_maxima_1d).
function _local_maxima(x::AbstractVector)
    n = length(x)
    mids = Int[]
    n < 3 && return mids
    i = 2
    i_max = n
    while i < i_max
        if x[i - 1] < x[i]
            i_ahead = i + 1
            while i_ahead < i_max && x[i_ahead] == x[i]
                i_ahead += 1
            end
            if x[i_ahead] < x[i]
                push!(mids, (i + (i_ahead - 1)) ÷ 2)
                i = i_ahead
            end
        end
        i += 1
    end
    return mids
end

# Topographic prominence of the peak at index p with no window limit
# (scipy.signal._peak_finding_utils._peak_prominences, wlen=None).
function _peak_prominence(x::AbstractVector, p::Int)
    n = length(x)
    left_min = x[p]
    i = p
    while i >= 1 && x[i] <= x[p]
        x[i] < left_min && (left_min = x[i])
        i -= 1
    end
    right_min = x[p]
    i = p
    while i <= n && x[i] <= x[p]
        x[i] < right_min && (right_min = x[i])
        i += 1
    end
    return x[p] - max(left_min, right_min)
end

"""
    find_peaks(x; prominence=nothing) -> (indices, prominences)

Indices (1-based) of local maxima of `x` whose prominence is at least
`prominence`, with their prominences -- the subset of
`scipy.signal.find_peaks` used by the original script.
"""
function find_peaks(x::AbstractVector{<:Real}; prominence = nothing)
    peaks = _local_maxima(x)
    proms = [_peak_prominence(x, p) for p in peaks]
    if prominence !== nothing
        keep = proms .>= prominence
        peaks, proms = peaks[keep], proms[keep]
    end
    return peaks, proms
end

# ---------------------------------------------------------------------------
# Ring / gap detection
# ---------------------------------------------------------------------------

"""
    find_rings_and_gaps(profile; ring_prominence=nothing, gap_prominence=nothing) -> (rings, gaps)

Locate local maxima (rings) and minima (gaps) in a radial flux profile with
`find_peaks`. `prominence=nothing` derives a threshold from the profile's own
scatter (3x the median `std_flux`); pass an explicit value to override if the
auto threshold is too strict/loose.
"""
function find_rings_and_gaps(profile; ring_prominence = nothing, gap_prominence = nothing)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]
    _dbg("find_rings_and_gaps: valid profile points =", length(flux_v), "/", length(profile.mean_flux))

    if ring_prominence === nothing
        ring_prominence = 3 * median(filter(!isnan, profile.std_flux))
    end
    gap_prominence === nothing && (gap_prominence = ring_prominence)
    _dbg("find_rings_and_gaps: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(flux_v; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-flux_v; prominence = gap_prominence)
    _dbg("find_rings_and_gaps: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    rings = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(ring_idx, ring_proms)]
    gaps  = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(gap_idx, gap_proms)]
    return rings, gaps
end

"""
    find_rings_and_gaps_residual(profile; window_frac=0.22, polyorder=3, ring_prominence=nothing,
                                 gap_prominence=nothing, prominence_sigma=1.2,
                                 min_radius_beams=2.5, min_abs_residual_frac=0.3)
        -> (rings, gaps, residual_data)

Locate ring/gap substructure using a smooth-baseline-subtraction approach
(as in e.g. Huang et al. 2018's DSHARP substructure analysis), rather than
`find_peaks` on the raw profile.

Why this is needed: `find_peaks` only flags genuine local maxima/minima.
When a disk's radial profile is dominated by a steep, monotonically
declining envelope (a bright unresolved central source falling off with
radius, as here), real ring/gap substructure often shows up only as
"shoulders" -- slope changes riding on top of the decline -- which never
turn over into a local max/min and so are invisible to plain `find_peaks`.
This function fits a smooth Savitzky-Golay baseline to the profile
(capturing just the broad monotonic falloff, not the substructure),
subtracts it to get a residual, and runs `find_peaks` on the residual.

window_frac: Savitzky-Golay window as a fraction of the number of valid
    profile bins (forced odd). Larger = smoother baseline = more sensitive
    to broad shoulders; smaller = baseline tracks the data more closely =
    less sensitive.
polyorder: polynomial order of the Savitzky-Golay fit (3 is a reasonable
    default for a smoothly declining profile with a few features).
prominence_sigma: multiplier on the residual's standard deviation used to
    derive ring/gap prominence thresholds when those are `nothing`. Lower
    values (e.g. 0.6-0.8) recover weaker, few-percent features (like the
    shallow gaps at ~6 au and ~44 au reported for TW Hya by Tsukagoshi et
    al. 2016 and Andrews et al. 2016) at the cost of more noise-driven false
    positives -- always sanity-check low-sigma detections against the plot
    and the literature rather than trusting them blindly.
min_radius_beams: exclude detections within this many synthesized-beam FWHMs
    (BMAJ) of the center. CLEAN deconvolution of a very bright, compact,
    high-dynamic-range core commonly leaves negative-bowl / restoration
    artifacts within the first few beam-widths of the peak, which can
    masquerade as a "gap". The baseline fit still uses the full profile;
    only the peak-finding *results* are filtered by radius. 0 disables.
min_abs_residual_frac: additionally require |residual| at the detected
    point to be at least this fraction of the residual's standard deviation.
    Prominence is relative (vs. neighbouring valleys/peaks), so a detection
    can be "prominent" while sitting right on the noise floor in absolute
    terms; this catches that case. 0 disables.
"""
function find_rings_and_gaps_residual(profile; window_frac = 0.22, polyorder = 3,
                                      ring_prominence = nothing, gap_prominence = nothing,
                                      prominence_sigma = 1.2, min_radius_beams = 2.5,
                                      min_abs_residual_frac = 0.3)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]

    n = length(flux_v)
    window = max(round(Int, window_frac * n), polyorder + 2)
    iseven(window) && (window += 1)
    window = min(window, isodd(n) ? n : n - 1)
    _dbg("find_rings_and_gaps_residual: n_valid =", n, "savgol window =", window, "polyorder =", polyorder)

    baseline = savgol_filter(flux_v, window, polyorder)
    residual = flux_v .- baseline
    residual_std = std(residual; corrected = false)          # population std, like np.std
    _dbg("find_rings_and_gaps_residual: residual std =", residual_std,
         "residual min/max =", minimum(residual), maximum(residual))

    ring_prominence === nothing && (ring_prominence = prominence_sigma * residual_std)
    gap_prominence === nothing && (gap_prominence = prominence_sigma * residual_std)
    _dbg("find_rings_and_gaps_residual: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(residual; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-residual; prominence = gap_prominence)
    _dbg("find_rings_and_gaps_residual: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    bmaj_au = profile.bmaj_au
    min_radius_au = (bmaj_au !== nothing && min_radius_beams > 0) ? min_radius_beams * bmaj_au : 0.0
    _dbg("find_rings_and_gaps_residual: bmaj_au =", bmaj_au, "min_radius_beams =", min_radius_beams,
         "-> min_radius_au =", min_radius_au)

    min_abs_residual = min_abs_residual_frac * residual_std
    _dbg("find_rings_and_gaps_residual: min_abs_residual_frac =", min_abs_residual_frac,
         "-> min_abs_residual =", min_abs_residual)

    keep(i) = radius_v[i] >= min_radius_au && abs(residual[i]) >= min_abs_residual

    all_idx = vcat(ring_idx, gap_idx)
    dropped_radius = [i for i in all_idx if radius_v[i] < min_radius_au]
    dropped_noise = [i for i in all_idx if radius_v[i] >= min_radius_au && abs(residual[i]) < min_abs_residual]
    if !isempty(dropped_radius)
        @printf("Excluded %d detection(s) within %.2f au (%sx beam FWHM) of center as likely beam/CLEAN artifacts:\n",
                length(dropped_radius), min_radius_au, min_radius_beams)
        for i in dropped_radius
            @printf("    r = %.1f au, residual = %.4e Jy/beam (%+.2f sigma)\n",
                    radius_v[i], residual[i], residual[i] / residual_std)
        end
    end
    if !isempty(dropped_noise)
        @printf("Excluded %d detection(s) with |residual| below %.2e Jy/beam (%sx residual std) as likely noise/boundary artifacts, despite passing the prominence test:\n",
                length(dropped_noise), min_abs_residual, min_abs_residual_frac)
        for i in dropped_noise
            @printf("    r = %.1f au, residual = %.4e Jy/beam (%+.2f sigma)\n",
                    radius_v[i], residual[i], residual[i] / residual_std)
        end
    end

    rings = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
             for (i, p) in zip(ring_idx, ring_proms) if keep(i)]
    gaps = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
            for (i, p) in zip(gap_idx, gap_proms) if keep(i)]
    residual_data = (radius_au = radius_v, flux = flux_v, baseline = baseline, residual = residual,
                     min_radius_au = min_radius_au, min_abs_residual = min_abs_residual,
                     distance_pc = profile.distance_pc)
    return rings, gaps, residual_data
end

"""
    report_residual_near(residual_data, target_radius_au; window_au=3.0)

Print the raw residual value(s) near a specific radius, independent of
whether `find_peaks` flagged anything there. Useful for checking a feature
that a prominence/noise-floor filter may have dropped (or that never
registered as a local extremum): "is there a real few-sigma dip near the
literature's 44 au gap even though nothing survived filtering there?" No
threshold is applied; it just reports what the residual actually does.
"""
function report_residual_near(residual_data, target_radius_au; window_au = 3.0)
    r_au = residual_data.radius_au
    res = residual_data.residual
    res_std = std(res; corrected = false)

    mask = abs.(r_au .- target_radius_au) .<= window_au
    if !any(mask)
        println("No profile bins found within $window_au au of $target_radius_au au.")
        return nothing
    end

    @printf("\nRaw residual near r = %s au (+/-%s au, residual std = %.4e Jy/beam):\n",
            target_radius_au, window_au, res_std)
    rm_, resm = r_au[mask], res[mask]
    dmin = minimum(abs.(rm_ .- target_radius_au))
    for (r, v) in zip(rm_, resm)
        marker = abs(r - target_radius_au) == dmin ? " <-- closest" : ""
        @printf("    r = %.2f au, residual = %.4e Jy/beam (%+.2f sigma)%s\n", r, v, v / res_std, marker)
    end

    ci = argmin(abs.(r_au .- target_radius_au))
    @printf("  Closest bin: r = %.2f au, residual = %.4e Jy/beam (%+.2f sigma)\n",
            r_au[ci], res[ci], res[ci] / res_std)
    return nothing
end

## Plots

In [ ]:
# ---------------------------------------------------------------------------
# Radial profile plots
# ---------------------------------------------------------------------------

"""
    plot_radial_profile(profile; rings=[], gaps=[], title="TW Hydrae radial profile") -> Figure

Plot azimuthally averaged flux vs. radius, with rings/gaps marked.
"""
function plot_radial_profile(profile; rings = [], gaps = [], title = "TW Hydrae radial profile",
                             filename = "twhya_radial_profile.png")
    _dbg("plot_radial_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r = profile.radius_au
    f = profile.mean_flux
    s = profile.std_flux

    fig = Figure(size = (820, 520))
    ax = Axis(fig[1, 1]; xlabel = "Deprojected radius [au]", ylabel = "Flux density [Jy/beam]",
              title = title, titlesize = 13)
    band!(ax, r, f .- s, f .+ s; color = (C_FLUX, 0.2), label = "±1σ scatter")
    lines!(ax, r, f; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax, rr, [x.flux for x in rings]; color = C_RING, marker = :utriangle,
                 markersize = 13, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax, gr, [x.flux for x in gaps]; color = C_GAP, marker = :dtriangle,
                 markersize = 13, label = "Gap")
    end
    axislegend(ax; position = :rt)

    _dbg("plot_radial_profile: about to display")
    display(fig)
    _save_fig(fig, filename)
    _dbg("plot_radial_profile: display returned")
    return fig
end

"""
    plot_residual_profile(residual_data; rings=[], gaps=[], title=..., show_literature=true) -> Figure

Two-panel plot: top shows the raw radial profile with its fitted
Savitzky-Golay baseline overlaid; bottom shows the residual (raw - baseline)
with detected rings (^) and gaps (v) marked. This is the view that actually
reveals substructure hidden inside a steep monotonic decline -- see
`find_rings_and_gaps_residual` for why the raw-profile view alone can miss it.

show_literature: if true and `residual_data` carries a `distance_pc`, overlay
    literature gap/hole radii from Tsukagoshi et al. 2016 / Andrews et al.
    2016 (see `literature_features_au`), rescaled to this pipeline's assumed
    distance, as dotted vertical lines -- so every run is visually
    self-checking against published values. TW Hya specific; set false for
    other targets.
"""
function plot_residual_profile(residual_data; rings = [], gaps = [],
                               title = "TW Hydrae substructure (baseline-subtracted)",
                               show_literature = true, filename = "twhya_substructure_residual.png")
    _dbg("plot_residual_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r = residual_data.radius_au
    min_radius_au = residual_data.min_radius_au
    distance_pc = residual_data.distance_pc

    fig = Figure(size = (820, 820))
    ax1 = Axis(fig[1, 1]; ylabel = "Flux density [Jy/beam]", title = title, titlesize = 13)
    ax2 = Axis(fig[2, 1]; xlabel = "Deprojected radius [au]", ylabel = "Residual [Jy/beam]")
    linkxaxes!(ax1, ax2)
    hidexdecorations!(ax1; grid = false)
    rowsize!(fig.layout, 1, Auto(1.2))
    rowsize!(fig.layout, 2, Auto(1.0))
    rowgap!(fig.layout, 8)

    lines!(ax1, r, residual_data.flux; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")
    lines!(ax1, r, residual_data.baseline; color = "#CCCCCC", linewidth = 1.4, linestyle = :dash,
           label = "Smooth baseline")
    if min_radius_au > 0
        vspan!(ax1, [0.0], [min_radius_au]; color = ("#888888", 0.22),
               label = @sprintf("Excluded (<%.1f au, beam/CLEAN artifacts)", min_radius_au))
        vspan!(ax2, [0.0], [min_radius_au]; color = ("#888888", 0.22))
    end

    hlines!(ax2, 0.0; color = "#888888", linewidth = 0.8)
    lines!(ax2, r, residual_data.residual; color = C_RESID, linewidth = 2, label = "Residual")

    if show_literature && distance_pc !== nothing
        lit = literature_features_au(distance_pc)
        _dbg("plot_residual_profile: literature features (au) =", lit)
        style = Dict(:hole => (color = ("#AAAAAA", 0.7), linewidth = 1.2),
                     :deep => (color = (C_LIT, 0.85), linewidth = 1.6),
                     :shallow => (color = (C_LIT, 0.55), linewidth = 1.1))
        for kind in (:hole, :deep, :shallow)
            sel = [x for x in lit if x[3] == kind]
            isempty(sel) && continue
            for ax in (ax1, ax2)
                vlines!(ax, [x[1] for x in sel]; linestyle = :dot, label = sel[1][2], style[kind]...)
            end
        end
    end

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax2, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax2, rr, [x.residual for x in rings]; color = C_RING, marker = :utriangle,
                 markersize = 15, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax2, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax2, gr, [x.residual for x in gaps]; color = C_GAP, marker = :dtriangle,
                 markersize = 15, label = "Gap")
    end

    axislegend(ax1; position = :rt, labelsize = 11)
    axislegend(ax2; position = :rt, labelsize = 11)

    _dbg("plot_residual_profile: about to display")
    display(fig)
    _save_fig(fig, filename)
    _dbg("plot_residual_profile: display returned")
    return fig
end

## Pipeline

In [ ]:
# ---------------------------------------------------------------------------
# Full pipeline (equivalent of the Python script's `if __name__ == "__main__"`)
# ---------------------------------------------------------------------------

# Prefer a continuum / pbcor image if multiple FITS files were extracted.
function _rank_fits(f)
    lf = lowercase(f)
    occursin("cont", lf) && occursin("pbcor", lf) && return 0
    occursin("cont", lf) && return 1
    occursin("pbcor", lf) && return 2
    return 3
end

"""
    run_twhya_pipeline(; proposal_id="2015.1.00686.S", ...) -> NamedTuple

Target the well-known Andrews et al. 2016 high-resolution Band 7 continuum
project (set `proposal_id=nothing` to browse all ~80 TW Hya observations and
pick a different `member_ous_uid` by hand). Returns everything produced so
individual steps can be re-run without repeating the download.
"""
function run_twhya_pipeline(; proposal_id = "2015.1.00686.S", distance_pc = 60.1,
                            incl_deg = 7.0, pa_deg = 155.0, max_radius_arcsec = 2.5, n_bins = 120,
                            prominence_sigma = 0.7, cutout_arcsec = 3.0, asinh_a = 0.01,
                            vmax_percentile = 99.5)
    results = query_tw_hya(; proposal_id = proposal_id)
    if nrow(results) == 0
        println("No rows matched that proposal_id. Re-run with proposal_id=nothing " *
                "to see all available TW Hya projects and pick a member_ous_uid " *
                "from the printed table.")
        return nothing
    end

    # Take the first matching member OUS (a proposal can have multiple). The
    # 4 rows returned for 2015.1.00686.S are per-spectral-window entries that
    # all share the same member_ous_uid, which is expected.
    member_ous_uid = String(strip(results.member_ous_uid[1]))

    data_info = inspect_products(member_ous_uid)

    tarball_url = pick_product_tarball(data_info)
    if tarball_url === nothing
        println("Could not find a usable product tarball for this UID. Print " *
                "data_info.access_url in full and pick one manually, or try " *
                "a different member_ous_uid / proposal_id.")
        return (results = results, data_info = data_info)
    end

    extracted_files = download_and_extract_fits(tarball_url)
    if isempty(extracted_files)
        println("Download succeeded but no FITS files were extracted -- this " *
                "tarball may not contain FITS images (e.g. it's a raw ASDM or " *
                "measurement-set-only package). Try the external_ari_l tarball " *
                "instead, or inspect the tar contents manually.")
        return (results = results, data_info = data_info, tarball_url = tarball_url)
    end

    println("\nExtracted $(length(extracted_files)) FITS file(s):")
    foreach(f -> println("  ", f), extracted_files)

    local_path = sort(extracted_files; by = _rank_fits, alg = MergeSort)[1]
    println("\nUsing: ", local_path)

    # High-resolution Band 7 data resolve sub-arcsec rings around a disk
    # that's only ~1-2" across, so crop tight and stretch hard.
    display_fits_image(local_path; cutout_arcsec = cutout_arcsec, asinh_a = asinh_a,
                       vmax_percentile = vmax_percentile)

    profile = rings = gaps = rings_r = gaps_r = residual_data = nothing

    # Azimuthally averaged radial profile to pull out ring/gap locations
    # numerically, rather than eyeballing them off the 2D image. TW Hya is
    # nearly face-on (i ~ 7 deg) so deprojection has little effect here.
    _dbg("main: entering radial profile block, local_path =", local_path)
    try
        profile = extract_radial_profile(local_path; incl_deg = incl_deg, pa_deg = pa_deg,
                                         distance_pc = distance_pc,
                                         max_radius_arcsec = max_radius_arcsec, n_bins = n_bins)
        rings, gaps = find_rings_and_gaps(profile)

        println("\nRadial profile center (pixel, 1-based): ", profile.center_pix)
        @printf("Pixel scale: %.4f arcsec/pix\n", profile.pix_scale_arcsec)

        println("\nDetected $(length(rings)) ring(s):")
        for r in rings
            @printf("  r = %.1f au, flux = %.4e Jy/beam, prominence = %.2e\n", r.radius_au, r.flux, r.prominence)
        end
        println("\nDetected $(length(gaps)) gap(s):")
        for g in gaps
            @printf("  r = %.1f au, flux = %.4e Jy/beam, prominence = %.2e\n", g.radius_au, g.flux, g.prominence)
        end

        plot_radial_profile(profile; rings = rings, gaps = gaps,
            title = "TW Hydrae Band 7 continuum radial profile — $(basename(local_path))")

        # Raw find_peaks on the profile finds 0 rings/0 gaps because the flux
        # declines monotonically overall -- real substructure shows up as
        # shoulders on that decline, not local max/min. Fit out the smooth
        # envelope and look at the residual instead. Literature (Tsukagoshi
        # et al. 2016; Andrews et al. 2016) reports two additional weak
        # (few-%) gaps at ~6 au and ~44 au beyond the two strong ones, so use
        # a low prominence threshold (0.7 sigma) to try to pull those out too.
        rings_r, gaps_r, residual_data = find_rings_and_gaps_residual(profile; prominence_sigma = prominence_sigma)
        println("\n[baseline-subtracted] Detected $(length(rings_r)) ring(s):")
        for r in rings_r
            @printf("  r = %.1f au, residual = %.4e Jy/beam, prominence = %.2e\n", r.radius_au, r.residual, r.prominence)
        end
        println("\n[baseline-subtracted] Detected $(length(gaps_r)) gap(s):")
        for g in gaps_r
            @printf("  r = %.1f au, residual = %.4e Jy/beam, prominence = %.2e\n", g.radius_au, g.residual, g.prominence)
        end

        # The literature's shallow ~44 au gap (scaled to our distance_pc) can
        # land near a spot that gets filtered out by min_abs_residual_frac.
        # Rather than loosening that threshold globally (risking noise
        # elsewhere), check the raw, unfiltered residual there directly.
        lit_shallow_44au_scaled = 44.0 * (profile.distance_pc / TSUKAGOSHI2016_TWHYA.distance_pc)
        report_residual_near(residual_data, lit_shallow_44au_scaled; window_au = 4.0)

        plot_residual_profile(residual_data; rings = rings_r, gaps = gaps_r,
            title = "TW Hydrae Band 7 continuum substructure — $(basename(local_path))")
        _dbg("main: radial profile block completed without exception")
    catch err
        println("ERROR: radial profile block raised an exception:")
        showerror(stdout, err, catch_backtrace())
        println()
    end

    return (results = results, data_info = data_info, tarball_url = tarball_url,
            fits_files = extracted_files, local_path = local_path, profile = profile,
            rings = rings, gaps = gaps, rings_residual = rings_r, gaps_residual = gaps_r,
            residual_data = residual_data)
end

## Run
Downloads the product tarball on first run (can be several GB; only the FITS files are kept). Plots are shown inline and saved to `./alma_twhya_plots/`.

In [ ]:
out = run_twhya_pipeline(proposal_id = "2015.1.00686.S");

## Optional: re-tune the substructure detection

In [ ]:
# Re-run just the substructure detection with different settings -- no
# re-download or re-reading of the FITS file needed.
if out !== nothing && out.profile !== nothing
    rings2, gaps2, rd2 = find_rings_and_gaps_residual(out.profile; prominence_sigma = 0.6,
                                                       window_frac = 0.22, min_radius_beams = 2.5)
    plot_residual_profile(rd2; rings = rings2, gaps = gaps2,
        title = "TW Hydrae substructure (prominence_sigma = 0.6)",
        filename = "twhya_substructure_residual_sigma0p6.png");
end